In [ ]:
import numpy as np
from PIL import Image
import cv2
radii = np.linspace(5, 150, 30)


DATA_ROOT = "dataset"  

SEQ = "seq1"
IMG_A = f"{DATA_ROOT}/{SEQ}/img1.ppm"
IMG_B = f"{DATA_ROOT}/{SEQ}/img2.ppm"
H12   = f"{DATA_ROOT}/{SEQ}/H1to2p"

TOP_K  = 5000    # ORB-style opencv
EPS_PX = 3.0     # typical repeatability radius
K_HARRIS = 0.04
THRESH = 1e6     


In [ ]:
def load_gray_u8(path: str) -> np.ndarray:
    img = Image.open(path).convert("L")
    return np.array(img, dtype=np.uint8)

def load_homography(path: str) -> np.ndarray:
    # file is 3x3 floats
    H = []
    with open(path, "r") as f:
        for line in f:
            H.extend([float(x) for x in line.strip().split()])
    return np.array(H, dtype=np.float32).reshape(3, 3)

imgA = load_gray_u8(IMG_A)
imgB = load_gray_u8(IMG_B)
H_A2B = load_homography(H12)

print("imgA:", imgA.shape, imgA.dtype, "min/max:", imgA.min(), imgA.max())
print("imgB:", imgB.shape, imgB.dtype, "min/max:", imgB.min(), imgB.max())
print("H:\n", H_A2B)


In [ ]:
def harris(img_u8: np.ndarray, k: float = 0.04) -> np.ndarray:
    """
    Match your HW:
    - central gradients
    - Gaussian weights g1d (7 taps)
    - accumulate over inner 5x5 using weights g1d[1..5]
    - valid centers are [3..H-4], [3..W-4]
    """
    img = img_u8.astype(np.float32)
    H, W = img.shape

    # central gradients (same idea as your HW gx, gy)
    gx = np.zeros((H, W), dtype=np.float32)
    gy = np.zeros((H, W), dtype=np.float32)
    gx[:, 1:-1] = img[:, 2:] - img[:, :-2]
    gy[1:-1, :] = img[2:, :] - img[:-2, :]

    gx2 = gx * gx
    gy2 = gy * gy
    gxy = gx * gy

    g1d = np.array([
        0.0044330482, 0.0540055826, 0.2420362234, 0.3990502954,
        0.2420362234, 0.0540055826, 0.0044330482
    ], dtype=np.float32)

    # inner 5 weights (indices 1..5) -> 5x5 separable kernel
    g5 = g1d[1:6]
    w5 = np.outer(g5, g5).astype(np.float32)

    # weighted sums over 5x5 neighborhood (vectorized via shifted slices)
    pad = 2
    gx2p = np.pad(gx2, ((pad, pad), (pad, pad)), mode="constant", constant_values=0)
    gy2p = np.pad(gy2, ((pad, pad), (pad, pad)), mode="constant", constant_values=0)
    gxyp = np.pad(gxy, ((pad, pad), (pad, pad)), mode="constant", constant_values=0)

    Ixx = np.zeros((H, W), dtype=np.float32)
    Iyy = np.zeros((H, W), dtype=np.float32)
    Ixy = np.zeros((H, W), dtype=np.float32)

    for dy in range(5):
        for dx in range(5):
            w = w5[dy, dx]
            Ixx += w * gx2p[dy:dy+H, dx:dx+W]
            Iyy += w * gy2p[dy:dy+H, dx:dx+W]
            Ixy += w * gxyp[dy:dy+H, dx:dx+W]

    detM = Ixx * Iyy - Ixy * Ixy
    trace = Ixx + Iyy
    R = detM - np.float32(k) * trace * trace

    # match your "valid center only" logic (7x7 needs 3-pixel margin)
    valid = np.zeros((H, W), dtype=bool)
    valid[3:H-3, 3:W-3] = True

    out = np.zeros((H, W), dtype=np.float32)
    out[valid] = R[valid]
    return out

scoreA = harris(imgA)
scoreB = harris(imgB)

print("scoreA min/max:", float(scoreA.min()), float(scoreA.max()))


In [ ]:
def nms5x5_strict_topk(score: np.ndarray, thresh: float, top_k: int):
    """
    Strict 5x5 NMS:
    keep center if score > thresh and strictly > all 24 neighbors.
    Return (x,y,score) sorted by score desc, truncated to top_k.
    """
    H, W = score.shape
    pad = 2

    sp = np.pad(score, ((pad, pad), (pad, pad)),
                mode="constant", constant_values=-np.inf)

    center = sp[pad:pad+H, pad:pad+W]
    max_nb = np.full((H, W), -np.inf, dtype=np.float32)

    for dy in range(-2, 3):
        for dx in range(-2, 3):
            if dy == 0 and dx == 0:
                continue
            nb = sp[pad+dy:pad+dy+H, pad+dx:pad+dx+W]
            max_nb = np.maximum(max_nb, nb)

    is_max = (center > max_nb) & (center > thresh)

    # keep only valid centers (avoid borders)
    valid = np.zeros((H, W), dtype=bool)
    valid[3:H-3, 3:W-3] = True
    is_max &= valid

    ys, xs = np.where(is_max)
    sc = score[ys, xs]

    if len(sc) == 0:
        return np.zeros((0, 3), dtype=np.float32)

    # sort by score desc
    order = np.argsort(-sc)
    xs = xs[order]
    ys = ys[order]
    sc = sc[order]

    n = min(top_k, len(sc))
    out = np.stack([xs[:n], ys[:n], sc[:n]], axis=1).astype(np.float32)
    return out

kpA = nms5x5_strict_topk(scoreA, THRESH, TOP_K)
kpB = nms5x5_strict_topk(scoreB, THRESH, TOP_K)

print("kpA:", kpA.shape[0], "kpB:", kpB.shape[0])
print("top score A:", float(kpA[0,2]) if len(kpA) else None)


In [ ]:
def project_points_xy(pts_xy: np.ndarray, H: np.ndarray) -> np.ndarray:
    """
    pts_xy: (N,2) float
    returns projected (N,2)
    """
    if len(pts_xy) == 0:
        return np.zeros((0,2), dtype=np.float32)

    pts = np.ones((len(pts_xy), 3), dtype=np.float32)
    pts[:, 0] = pts_xy[:, 0]
    pts[:, 1] = pts_xy[:, 1]

    q = (H @ pts.T).T
    q[:, 0] /= q[:, 2]
    q[:, 1] /= q[:, 2]
    return q[:, :2].astype(np.float32)

def repeatability(kp1_xy: np.ndarray, kp2_xy: np.ndarray, H12: np.ndarray, eps: float, H2: int, W2: int):
    """
    Detector repeatability:
    project kp1 -> image2, count how many have a kp2 within eps pixels.
    repeatability = matches / min(N1, N2)
    returns mean localization error of matched ones
    """
    N1 = len(kp1_xy)
    N2 = len(kp2_xy)
    if N1 == 0 or N2 == 0:
        return 0.0, 0, 0.0

    proj = project_points_xy(kp1_xy.astype(np.float32), H12)

    # keep only projections inside image2 bounds
    inside = (proj[:,0] >= 0) & (proj[:,0] < W2) & (proj[:,1] >= 0) & (proj[:,1] < H2)
    proj = proj[inside]
    if len(proj) == 0:
        return 0.0, 0, 0.0

    # brute force nearest neighbor (fine for TOP_K=500)
    matches = 0
    err_sum = 0.0

    for p in proj:
        dx = kp2_xy[:,0] - p[0]
        dy = kp2_xy[:,1] - p[1]
        d2 = dx*dx + dy*dy
        j = int(np.argmin(d2))
        d = float(np.sqrt(d2[j]))
        if d <= eps:
            matches += 1
            err_sum += d

    rep = matches / float(min(N1, N2))
    mean_err = (err_sum / matches) if matches > 0 else 0.0
    return rep, matches, mean_err

repA2B, mA2B, errA2B = repeatability(kpA[:, :2], kpB[:, :2], H_A2B, EPS_PX, imgB.shape[0], imgB.shape[1])

print("Repeatability A->B:", repA2B)
print("Matches:", mA2B)
print("Mean localization error (px):", errA2B)


In [ ]:
# Spatial Coverage Evaluation (Ripley’s K)
#Evaluation of the spatial coverage of detected keypoints using Ripley’s spatial statistics: Coverage is computed on keypoints restricted to the overlap region defined by the ground-truth tomography.


In [ ]:
def ripley_K(points, radii, window_area):
    """
    points: (N,2) array of point coordinates
    radii:  array of distances r
    window_area: area of the observation window

    returns: K(r) for each r
    """
    N = len(points)
    K = np.zeros(len(radii), dtype=np.float32)

    # pairwise squared distances
    dx = points[:, None, 0] - points[None, :, 0]
    dy = points[:, None, 1] - points[None, :, 1] #Compute distances between every pair of points
    dist2 = dx*dx + dy*dy #Squared Euclidean distance (faster, no sqrt)

    # ignore self-distances
    np.fill_diagonal(dist2, np.inf) #Removes self-counting (a point is not its own neighbor)

    for i, r in enumerate(radii):
        K[i] = window_area / (N*(N-1)) * np.sum(dist2 <= r*r) #dist2 <= r*r: Counts neighbors inside a circle of radius r 
        # window_area / (N*(N-1)):Density normalization(this is what makes it Ripley’s K)

    return K


In [ ]:
kpA   # shape (N,3): x, y, score
pts_A = kpA[:, :2]   # (x, y)
radii = np.linspace(5, 200, 30)

In [ ]:
proj = project_points_xy(pts_A, H_A2B)

inside = (
    (proj[:,0] >= 0) & (proj[:,0] < imgB.shape[1]) &
    (proj[:,1] >= 0) & (proj[:,1] < imgB.shape[0])
)

pts_overlap = pts_A[inside]


In [ ]:
import matplotlib.pyplot as plt

H_img, W_img = imgA.shape
area_img = H_img * W_img

K_real = ripley_K(pts_overlap, radii, area_img)
L_real = np.sqrt(K_real / np.pi) - radii

plt.figure(figsize=(6,4))
plt.plot(radii, L_real, '-o')
plt.axhline(0, color='k', linestyle='--')
plt.xlabel("r (pixels)")
plt.ylabel("L(r) - r")
plt.title("Ripley coverage – Oxford VGG (Image A)")
plt.grid(True)
plt.show()


In [ ]:
#HW Evaluation
def read_hw_kps_txt(path):
    xs, ys, ss = [], [], []
    with open(path, "r") as f:
        for line in f:
            line = line.strip()
            if (not line) or line.startswith("#"):
                continue
            parts = line.split()
            if len(parts) < 3:
                continue
            x, y, s = float(parts[0]), float(parts[1]), float(parts[2])
            xs.append(x); ys.append(y); ss.append(s)

    xy = np.stack([xs, ys], axis=1).astype(np.float32)
    sc = np.array(ss, dtype=np.float32)
    return xy, sc

kpA_xy_hw, kpA_sc_hw = read_hw_kps_txt("seq1_nms_kps.txt")
kpB_xy_hw, kpB_sc_hw = read_hw_kps_txt("seq1_nms_kps2.txt")

print("HW img1:", kpA_xy_hw.shape, "HW img2:", kpB_xy_hw.shape)
print("score img1 min/max:", kpA_sc_hw.min(), kpA_sc_hw.max())
print("score img2 min/max:", kpB_sc_hw.min(), kpB_sc_hw.max())

In [ ]:
rep_hw, matches_hw, err_hw = repeatability(
    kpA_xy_hw, kpB_xy_hw, H_A2B, EPS_PX, imgB.shape[0], imgB.shape[1]
)

print("Repeatability HW A->B:", rep_hw)
print("Matches HW:", matches_hw)
print("Mean localization error HW (px):", err_hw)
